# Rough Volatility Models
## 1. Stochastic Volatility Models

### 1.1 The Limitations of Black-Scholes
The starting point of modern option pricing is the Black-Scholes (BS) model. While revolutionary, it relies on a strong assumption that contradicts market reality.

**The Constant Volatility Assumption**
In the BS world, the underlying asset price $S_t$ follows a Geometric Brownian Motion:

$$dS_t = \mu S_t dt + \sigma S_t dW_t$$

Here, $\sigma$ (volatility) is assumed to be a **constant**.

* **The Implication:** If the BS model were perfectly correct, the implied volatility derived from market prices of options on the same asset should be identical, regardless of the strike price ($K$) or time to maturity ($T$).
* **The Reality:** This is rarely observed in real markets.

---

### 1.2 Market Reality: The Volatility Smile and Surface
Real-world data "punches back" against the constant volatility assumption. When we invert the BS formula using market prices to solve for $\sigma$, we find that $\sigma$ is not constant.

#### 1.2.1 The Volatility Smile
If we plot Implied Volatility (y-axis) against Strike Price $K$ (x-axis) for a fixed maturity:
* We do not see a flat line.
* We typically see a curve where volatility is lower for At-the-Money (ATM) options and higher for In-the-Money (ITM) or Out-of-the-Money (OTM) options.
* This shape resembles a smile, hence the term **Volatility Smile**.

#### 1.2.2 The Volatility Surface
The "Smile" only accounts for the strike dimension. If we add **Time to Maturity ($T$)** as a third dimension, we get the **Volatility Surface**.
* **X-axis:** Strike ($K$) or Moneyness.
* **Y-axis:** Time to Maturity ($T$).
* **Z-axis:** Implied Volatility ($\sigma$).

**Visual Intuition:** It looks like a "flying carpet." The near-term part of the surface is often jagged and steep (high skew), while the long-term part is usually flatter and smoother.

### 1.2.3 Realized vs. Implied Volatility
It is crucial to distinguish between two concepts:
1. **Realized Volatility (RV):** "Looking Backward." It is the standard deviation of historical returns over a past period (e.g., past 30 days). It is a statistical measurement of what *actually happened*.
2. **Implied Volatility (IV):** "Looking Forward." It is backed out from option prices. It represents the market's expectation of future volatility (and the price of volatility risk).

---

### 1.3 Measuring Realized Volatility
How do we calculate the volatility of a stock given a set of historical prices $\{P_0, P_1, ..., P_n\}$?

**Step 1: Log Returns**
Calculate the natural log returns of the price series:

$$r_i = \ln \left( \frac{P_i}{P_{i-1}} \right)$$

**Step 2: Standard Deviation**
Calculate the standard deviation of these returns. First find the mean $\bar{r}$, then:

$$\text{std} = \sqrt{\frac{\sum(r_i - \bar{r})^2}{n-1}}$$

**Step 3: Annualization**
Volatility is typically quoted in annualized terms. Standard deviation scales with the square root of time. If using daily data (assuming 252 trading days/year):

$$\text{Volatility}_{\text{annualized}} = \text{std} \times \sqrt{252}$$

---

### 1.4 The Stochastic Volatility Framework
To address the Volatility Smile, we move from deterministic volatility to **Stochastic Volatility (SV)**. We follow the framework closely related to Wilmott (2000).

We assume the stock price $S$ and its **variance** $v$ (where volatility $\sigma = \sqrt{v}$) satisfy the following system of Stochastic Differential Equations (SDEs):

#### The System of SDEs

1. **Asset Process:**
   $$dS_t = \mu_t S_t dt + \sqrt{v_t} S_t dZ_1 \quad (1.1)$$

2. **Variance Process:**
   $$dv_t = \alpha(S_t, v_t, t)dt + \eta \beta(S_t, v_t, t)\sqrt{v_t} dZ_2 \quad (1.2)$$

**Key Parameters:**
* $\mu_t$: Instantaneous drift of the stock.
* $\alpha(...)$: Drift of the variance process (e.g., mean reversion).
* $\eta$: The "volatility of volatility" (vol-of-vol).
* **Correlation ($\rho$):** The Wiener processes are correlated:
  $$\langle dZ_1, dZ_2 \rangle = \rho dt.$$

---

### 1.5 Hedging and Deriving the PDE
This is the core of the financial engineering approach. In Black-Scholes, we only have risk from $S$. In SV models, we have risk from both $S$ and $v$.

#### 1.5.1 The Hedging Problem
* **Black-Scholes:** Randomness comes from $dZ_1$ only. We can hedge an option $V$ simply by trading the underlying stock $S$.
* **Stochastic Volatility:** Randomness comes from $dZ_1$ and $dZ_2$.
    * We can hedge $S$-risk using the stock.
    * **Problem:** Volatility is not a traded asset. We cannot "buy variance" directly to hedge the $dv$ term.
    * **Solution:** We must use **another option** (a benchmark instrument) to hedge the volatility risk.

#### 1.5.2 Constructing the Portfolio
We set up a portfolio $\Pi$ containing:
1. One unit of the Option being priced: $V(S, v, t)$.
2. $-\Delta$ units of the Stock $S$.
3. $-\Delta_1$ units of another asset $V_1$ (e.g., a liquid option) to hedge volatility risk.

$$\Pi = V - \Delta S - \Delta_1 V_1$$

#### 1.5.3 Dynamics of the Portfolio
The change in the portfolio value $d\Pi$ is derived using **Itô's Lemma** (details omitted for brevity). The result groups terms by $dt, dS,$ and $dv$:

$$d\Pi = \{ \dots \} dt + \left( \frac{\partial V}{\partial S} - \Delta_1 \frac{\partial V_1}{\partial S} - \Delta \right) dS + \left( \frac{\partial V}{\partial v} - \Delta_1 \frac{\partial V_1}{\partial v} \right) dv$$

#### 1.5.4 Making the Portfolio Risk-Free
To make the portfolio instantaneously risk-free, we must eliminate all stochastic terms ($dS$ and $dv$).

1. **Eliminate $dv$ (Vega Risk):**
   $$\frac{\partial V}{\partial v} - \Delta_1 \frac{\partial V_1}{\partial v} = 0 \implies \Delta_1 = \frac{\partial V / \partial v}{\partial V_1 / \partial v}$$
   *This tells us how many units of the hedging option $V_1$ we need.*

2. **Eliminate $dS$ (Delta Risk):**
   $$\frac{\partial V}{\partial S} - \Delta_1 \frac{\partial V_1}{\partial S} - \Delta = 0$$
   *This tells us the net Delta position required in the stock.*

#### 1.5.5 The General PDE
Once the random terms are removed, the portfolio earns the risk-free rate $r$:
$$d\Pi = r \Pi dt$$

Substituting the expressions back, we find that the equation holds for any derivative only if both sides equal a function of the independent variables. This introduces the **Market Price of Volatility Risk**, often denoted as part of the term $(\alpha - \phi \beta \sqrt{v})$.

Rearranging gives the fundamental **Partial Differential Equation for Stochastic Volatility**:

$$\frac{\partial V}{\partial t} + \frac{1}{2} v S^2 \frac{\partial^2 V}{\partial S^2} + \rho \eta v \beta S \frac{\partial^2 V}{\partial v \partial S} + \frac{1}{2} \eta^2 v \beta^2 \frac{\partial^2 V}{\partial v^2} + r S \frac{\partial V}{\partial S} - rV = -(\alpha - \phi \beta \sqrt{v}) \frac{\partial V}{\partial v}$$

**Interpretation of Terms:**
* $\frac{\partial V}{\partial t}$: Time decay (**Theta**).
* $\frac{1}{2}vS^2 \frac{\partial^2 V}{\partial S^2}$: **Gamma** (standard Black-Scholes term, but with stochastic $v$).
* $\rho \eta v \beta S \frac{\partial^2 V}{\partial v \partial S}$: **Vanna**. The effect of correlation between price and vol changes.
* $\frac{1}{2}\eta^2 v \beta^2 \frac{\partial^2 V}{\partial v^2}$: **Volga** (Vol of Vol). The convexity with respect to variance.
* **RHS Term:** Represents the drift of the variance, adjusted for the market price of risk.

By acknowledging that volatility is stochastic, we move from a world where we only hedge **Delta** (using stock) to a world where we must hedge both **Delta** (Direction) and **Vega** (Volatility). 

### 📚 References & Further Reading
For a deeper dive into the dynamics of the volatility surface and the practical implementation of these models, the following resource is highly recommended:

* **Gatheral, J. (2006).** *The Volatility Surface: A Practitioner's Guide*. Hoboken, NJ: John Wiley & Sons.
